###Notebook Goal

Create advanced business analytics:

- Customer Lifetime Value (CLV)
- RFM Analysis
- Running Revenue
- Monthly Growth
- Top Customers by Country
- Advanced Window Functions

In [0]:
#Step 1: Read Silver Table
from pyspark.sql import functions as F
from pyspark.sql.window import Window

silver_df = spark.table(
    "workspace.retail.silver_online_retail"
)

silver_df.createOrReplaceTempView(
    "retail_sales"
)

In [0]:
%sql
-- Analytics 1: Customer Lifetime Value (CLV)

CREATE OR REPLACE TEMP VIEW customer_lifetime_value AS

SELECT
,
    ROUND(SUM(revenue),2) AS lifetime_revenue,
    COUNT(DISTINCT Invoice) AS total_orders
FROM retail_sales
GROUP BY Customer_ID

In [0]:
%sql
--Check:

SELECT *
FROM customer_lifetime_value
ORDER BY lifetime_revenue DESC
LIMIT 20

Customer_ID,lifetime_revenue,total_orders
18102.0,580987.04,145
14646.0,528602.52,151
14156.0,313437.62,156
14911.0,291420.81,398
17450.0,244784.25,51
13694.0,195640.69,143
17511.0,172132.87,60
16446.0,168472.5,2
16684.0,147142.77,55
12415.0,144458.37,28


In [0]:
%sql
-- Analytics 2: Monthly Revenue Trend


CREATE OR REPLACE TEMP VIEW monthly_revenue AS

SELECT
    YEAR(invoice_timestamp) AS year,
    MONTH(invoice_timestamp) AS month,
    ROUND(SUM(revenue),2) AS revenue
FROM retail_sales
GROUP BY
    YEAR(invoice_timestamp),
    MONTH(invoice_timestamp)

In [0]:
%sql
-- Analytics 3: Running Revenue

-- Window Function

SELECT
    year,
    month,
    revenue,

    SUM(revenue) OVER(
        ORDER BY year,month
    ) AS running_revenue

FROM monthly_revenue

In [0]:
%sql
-- Analytics 4: Monthly Revenue Growth %
WITH revenue_cte AS
(
SELECT
    year,
    month,
    revenue,

    LAG(revenue)
    OVER(
        ORDER BY year,month
    ) AS previous_month_revenue

FROM monthly_revenue
)

SELECT
    *,
    ROUND(
        (
            revenue - previous_month_revenue
        ) /
        previous_month_revenue * 100,
        2
    ) AS growth_percentage

FROM revenue_cte

year,month,revenue,previous_month_revenue,growth_percentage
2009,12,683504.01,null,null
2010,1,555802.67,683504.01,-18.68
2010,2,504558.95,555802.67,-9.22
2010,3,696978.47,504558.95,38.14
2010,4,591982.0,696978.47,-15.06
2010,5,597833.38,591982.0,0.99
2010,6,636371.13,597833.38,6.45
2010,7,589736.17,636371.13,-7.33
2010,8,602224.6,589736.17,2.12
2010,9,829013.95,602224.6,37.66


In [0]:
%sql
-- Analytics 5: Top Customer Per Country


WITH customer_sales AS
(
SELECT
    Country,
    Customer_ID,
    SUM(revenue) AS total_sales
FROM retail_sales
GROUP BY
    1,2
)

SELECT *
FROM
(
SELECT
    *,
    DENSE_RANK() OVER(
        PARTITION BY Country
        ORDER BY total_sales DESC
    ) AS ranking
FROM customer_sales
)
WHERE ranking = 1

Country,Customer_ID,total_sales,ranking
Australia,12415.0,144458.36999999968,1
Austria,12360.0,4252.890000000001,1
Bahrain,12355.0,947.6100000000001,1
Belgium,12380.0,9676.299999999994,1
Brazil,12769.0,1143.6,1
Canada,17444.0,2940.04,1
Channel Islands,14936.0,12410.809999999963,1
Cyprus,12359.0,8873.389999999998,1
Czech Republic,12781.0,826.7400000000001,1
Denmark,13902.0,34095.259999999995,1


In [0]:
%sql
-- Analytics 6: RFM Analysis

-- This is a premium Data Engineering / Analytics project feature.

-- Recency

-- Days since last purchase

-- Frequency

-- Total orders

-- Monetary

-- Total spend

SELECT
    Customer_ID,

    DATEDIFF(
        MAX(invoice_timestamp),
        MIN(invoice_timestamp)
    ) AS recency,

    COUNT(DISTINCT Invoice) AS frequency,

    ROUND(
        SUM(revenue),
        2
    ) AS monetary

FROM retail_sales

GROUP BY Customer_ID

Customer_ID,recency,frequency,monetary
13085.0,581,8,2433.28
15362.0,290,2,613.08
18102.0,738,145,580987.04
18087.0,640,17,14761.52
13635.0,671,4,2999.16
17519.0,721,15,4774.17
13758.0,727,15,8698.13
16321.0,666,7,604.55
16167.0,324,5,1386.12
17865.0,708,46,26312.19


In [0]:
rfm_df = spark.sql("""
SELECT
    Customer_ID,

    DATEDIFF(
        MAX(invoice_timestamp),
        MIN(invoice_timestamp)
    ) AS recency,

    COUNT(DISTINCT Invoice) AS frequency,

    ROUND(
        SUM(revenue),
        2
    ) AS monetary

FROM retail_sales

GROUP BY Customer_ID
""")

rfm_df.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable(
"workspace.retail.gold_rfm_analysis"
)

In [0]:
%sql
-- Analytics 7: Top Selling Products by Country
WITH product_sales AS
(
SELECT
    Country,
    Description,
    SUM(revenue) AS revenue
FROM retail_sales
GROUP BY
    Country,
    Description
)

SELECT *
FROM
(
SELECT
    *,
    ROW_NUMBER() OVER(
        PARTITION BY Country
        ORDER BY revenue DESC
    ) AS rn
FROM product_sales
)
WHERE rn <= 5

Country,Description,revenue,rn
Australia,RABBIT NIGHT LIGHT,3375.84,1
Australia,REGENCY CAKESTAND 3 TIER,2930.7000000000003,2
Australia,RED TOADSTOOL LED NIGHT LIGHT,2464.2000000000007,3
Australia,DOLLY GIRL LUNCH BOX,2182.2,4
Australia,SET OF 6 SPICE TINS PANTRY DESIGN,2082.0,5
Austria,POSTAGE,3056.0,1
Austria,EDWARDIAN PARASOL NATURAL,654.0,2
Austria,EDWARDIAN PARASOL RED,582.6,3
Austria,EDWARDIAN PARASOL BLACK,582.5999999999999,4
Austria,EDWARDIAN PARASOL PINK,464.1,5
